# VMT Reduction Route Visualization

This script provides a means to visualize the routes from the VMT mode shift project for QA/QC purposes.  Start by running all the way through Part 1: Read and merge data.  This will result in a combined dataframe to support visualization that is equivalent to what is used in the feasibility analysis.  Part 2 defines the main visualization methods. In Part 3, the user can select a subset of routes using any number of queries on the main dataframe, then visualize either a selected or a random trip from that dataframe.  


## Part 1: Read and merge data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

import importlib
import route_mapper

from shapely.geometry import Point


In [2]:
import math
import keyring
import itertools
from ast import literal_eval

import folium
# from folium.plugins import Legend   # having trouble on the import, using branca instead
from folium.features import GeoJsonPopup

import branca
import keyring


In [3]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [4]:
%%html
<style>
.rendered_html p {
            font-size: 12px;
            font-family: Times New Roman, serif;
            text-align:justify}
</style>

In [6]:
# This is used to avoid hard-coding directories. 
# To set the directory, use the command prompt or a notebook you don't check in.  run:
# import keyring
# keyring.set_password("msp", "vmt_reduction_dir", <directory>)

# get base path for data
data_dir = keyring.get_password("msp", "vmt_reduction_dir")

In [7]:
# read in the data
df = pd.read_csv(data_dir + "/data_processed/tbi_cleaned.csv")

C:\Users\ger225\AppData\Local\Temp\ipykernel_36128\635806421.py:2: DtypeWarning: Columns (33,34,35,52,64,65,66,67,68,69,70) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_dir + "/data_processed/tbi_cleaned.csv")


In [8]:
# read path geopkg data
# parquet files take an order of magnitude less time to read
car = gpd.read_parquet(data_dir + "/Data_Processed/geodata/car_congestion.parquet")
bike = gpd.read_parquet(data_dir + "/Data_Processed/geodata/bike_lts.parquet")
transit = gpd.read_parquet(data_dir + "/Data_Processed/geodata/transit_trips.parquet")
walk = gpd.read_parquet(data_dir + "/Data_Processed/geodata/walk.parquet")

In [9]:
# grab the observed routes
observed =  gpd.read_parquet(data_dir + "/Data_Processed/geodata/observed_locations.parquet")

In [10]:
# calculate duration from start/end times
transit["start_time_dt"] = pd.to_datetime(transit["start_time"])
transit["end_time_dt"] = pd.to_datetime(transit["end_time"])

transit["duration"] = (transit["end_time_dt"] - transit["start_time_dt"]).apply(lambda x: x.seconds / 60)

In [11]:
# create a gdf for linked trips
# note: This is currently unused.  Come back to it if needed. 
agg_fns = {
    "trip_id": "first",
    "leg_index": "count",
    "start_time": "first",
    "end_time": "last",
    "origin_stop_id": "first", # throw away
    "origin_stop_name": "first",
    "destination_stop_id": "first",
    "destination_stop_name": "first",
    "route_id": "first", 
    "route_short_name": "first",
    "route_long_name": "first",
    "route_type": "first", 
    "leg_type": "first", # end throw away
    "start_time_dt": "first",
    "end_time_dt": "last",
    "duration": "sum"
}

transit_grouped = transit.dissolve(by="trip_id", aggfunc=agg_fns) # merges geometries in addition to aggregating the rest of the columns

## Step 2: Set up the mapper

See route_mapper.py

In [12]:
# reload is only needed if we make changes to the module
importlib.reload(route_mapper)

# create an object that stores the data and knows how to map it
mapper = route_mapper.RouteMapper(df, car, walk, bike, transit, transit_grouped, observed)

## Step 3: Map some trips

The user can query individual trips and specify a trip id, or if no argument is given, the mapper will choose a trip randomly.  An example of each is shown below.  You can simply re-execute these cells to map additional trips. 

In [13]:
# option 1 is just to map a randomly selected trip
mapper.map_trip()

TypeError: can only concatenate str (not "int") to str

In [ ]:
# or I can specify a specific trip ID
# note that the trip IDs are strings that look like lists
# this is to account for the linking of transit (and some car) legs into complete trips

mapper.map_trip('[1911348302020]')

In [ ]:
# by doing this, we can select any trip_id we want, or select a random trip from a sample.
# we query using the standard pandas syntax

# for example, if we want to pick a random trip that chooses transit:

transit_subsample = df[df['mode']=="Transit"]
trip_id = transit_subsample.sample(n=1)['trip_id'].iloc[0]
mapper.map_trip(trip_id)

In [ ]:
# we can do the same thing for other modes

bike_subsample = df[df['mode']=="Bike/Scooter"]
trip_id = bike_subsample.sample(n=1)['trip_id'].iloc[0]
mapper.map_trip(trip_id)

In [ ]:
# or if we want to pick just trips that are very long


subsample = df[df['duration']>90]
trip_id = subsample.sample(n=1)['trip_id'].iloc[0]
mapper.map_trip(trip_id)